# 03c — v2 lambda Sweep (encoder feature injection)

Trains v2 (encoder feature injection, 7 nutrients added to ingredient embeddings before GIN) across the lambda values used by v3 / v4.

**Runtime**: Colab free T4 GPU, ~2.5 hr.

**Steps**:
1. Runtime > Change runtime type > T4 GPU
2. Set `PROJECT_ROOT` (cell 4) if your Drive layout differs
3. Runtime > Run all

## 1. GPU + dependencies

In [ ]:
!nvidia-smi

In [ ]:
# torch_geometric needs to be matched with the installed torch wheel.
# Colab's default torch is recent enough that the generic install works.
!pip install -q torch_geometric pandas numpy matplotlib

## 2. Mount Drive (code + data both live in Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
PROJECT_ROOT = '/content/drive/MyDrive/CS471_project'
CODE_DIR     = f'{PROJECT_ROOT}/code'
DATA_DIR     = f'{PROJECT_ROOT}/data'
OUTPUT_DIR   = f'{PROJECT_ROOT}/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(CODE_DIR)
print(f'CWD        = {os.getcwd()}')
print(f'DATA_DIR   = {DATA_DIR}')
print(f'OUTPUT_DIR = {OUTPUT_DIR}')

## 3. Smoke test (optional)

In [ ]:
# 1-epoch smoke test (~30 sec on T4). Skip by changing False below.
RUN_SMOKE = True
if RUN_SMOKE:
    !python src/train_v2.py --max_epochs 1 --patience 1 --no_resume \
      --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR}/smoke_v2
    print('\n[smoke] OK')

## 4. Run lambda sweep

In [ ]:
!python src/run_lambda_sweep.py --variant v2 \
  --data_dir {DATA_DIR} --output_dir {OUTPUT_DIR} \
  --tau_percentile 0 \
  --lambdas 0 0.1 1 5 10

## 5. Quick look

In [ ]:
import pandas as pd
df = pd.read_csv(f'{OUTPUT_DIR}/sweep_summary_v2.csv')
show = ['lambda_h', 'MRR', 'Hit@10', 'sugar_sat_pct',
         'sodium_sat_pct', 'flavor_cos_mean']
print(df[[c for c in show if c in df.columns]].round(2).to_string(index=False))